# <center>C8: Model Optimization and Tuning</center>

## 1. Optimization Challenges

Training Deep Networks is hard. Aside from selecting the right architecture, we often face "pathologies" that stop learning.

### 1.1 The Vanishing Gradient Problem
In deep networks, gradients are multiplied via the Chain Rule during Backpropagation. If these gradients are small (< 1), they shrink exponentially as they go back to the first layers. The result? **Early layers stop learning**.

![Vanishing Gradient Diagram](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Faiml.com%2Fwp-content%2Fuploads%2F2023%2F11%2Fvanishing-and-exploding-gradient-1.png&f=1&nofb=1&ipt=04f8bc8be69fd32fa79ce4f35a8a2005f67168ce5a2cc616ada9229a432d392f)
![Vanishing Gradient Neural Diagram](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fmiro.medium.com%2Fv2%2Fresize%3Afit%3A716%2F1*-rPcuafUSo-SwZF121y_rQ.png&f=1&nofb=1&ipt=6df49acd41f35a191a1e7d93af1a71175672a8ef3348d6cd2d5201a081c09cb8)

*As gradients flow backward (right to left), they become thinner (smaller), leaving initial layers unchanged.*

### 1.2 The Dying ReLU Problem
ReLU ($max(0, x)$) is great, but if a neuron's weights are updated such that it always outputs 0 (negative input), it will **never activate again**. Its gradient is 0 forever. It is "dead".

![Dying ReLU](https://aiml.com/wp-content/uploads/2023/09/ReLU-Leaky-ReLU-ELU-1024x794.png)

**Solutions**:
-   **Leaky ReLU**: Allows a small gradient when $x < 0$ ($y = 0.01x$).
-   **He Initialization**: Smart weight initialization.

## 2. Model Validation Strategy

We use a **Validation Set** to proxy performance on unseen data. The mostly critical concept here is **Overfitting**.

### Early Stopping
We stop training when the Validation Loss stops decreasing, even if Training Loss is still going down.

![Early Stopping Curve](https://www.pinecone.io/_next/image/?url=https%3A%2F%2Fcdn.sanity.io%2Fimages%2Fvr8gru94%2Fproduction%2Fdfa532f08c7b0720c72306d327a82f78f7e17f5d-1000x843.png&w=2048&q=75)

*The "Sweet Spot" is where the gap between Training and Validation loss starts to widen.*

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop if val_loss doesn't improve for 3 epochs
early_stop = EarlyStopping(monitor='val_loss', patience=3, verbose=1)

# model.fit(..., callbacks=[early_stop])

## 3. Dropout Regularization

**Overfitting** is when a model performs great on training data but poorly on unseen data. It "memorizes" rather than "learns".

**Dropout** is a powerful regularization technique that randomly "drops" (sets to zero) a percentage of neurons during training. This forces the network to learn robust features and prevents it from relying too heavily on any single neuron.

![Dropout Visualization](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fwww.baeldung.com%2Fwp-content%2Fuploads%2Fsites%2F4%2F2020%2F05%2F2-1-2048x745-1.jpg&f=1&nofb=1&ipt=a11c3f79fa65d08341ebc34f2b667f43ffe55155331486d01a2bf9bf34103efb)

### How It Works
- During **training**: Randomly drop neurons with probability `p` (e.g., 0.3 means 30% dropout)
- During **inference**: Use all neurons (no dropout)

### Best Practices
- Start with 0.2-0.5 dropout rate
- Place after Dense layers, before activation (or after)
- Higher dropout for larger networks
- Don't use dropout in the output layer

In [ ]:
# Example: Comparing models with and without Dropout
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

# 1. Create a dataset prone to overfitting (high dimensionality, small dataset)
X, y = make_classification(n_samples=500, n_features=50, n_informative=30,
                           n_redundant=10, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")

# 2. Model WITHOUT Dropout
model_no_dropout = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(50,)),
    layers.Dense(64, activation='relu'),
    layers.Dense(32, activation='relu'),
    layers.Dense(1, activation='sigmoid')
])

model_no_dropout.compile(optimizer='adam',
                         loss='binary_crossentropy',
                         metrics=['accuracy'])

# 3. Model WITH Dropout
model_with_dropout = keras.Sequential([
    layers.Dense(128, activation='relu', input_shape=(50,)),
    layers.Dropout(0.3),  # Drop 30% of neurons
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])

model_with_dropout.compile(optimizer='adam',
                           loss='binary_crossentropy',
                           metrics=['accuracy'])

# 4. Train both models
print("\nTraining model WITHOUT dropout...")
history_no_dropout = model_no_dropout.fit(X_train, y_train,
                                          validation_data=(X_test, y_test),
                                          epochs=100,
                                          batch_size=32,
                                          verbose=0)

print("Training model WITH dropout...")
history_with_dropout = model_with_dropout.fit(X_train, y_train,
                                              validation_data=(X_test, y_test),
                                              epochs=100,
                                              batch_size=32,
                                              verbose=0)

# 5. Visualize the comparison
plt.figure(figsize=(14, 5))

# Training Accuracy
plt.subplot(1, 2, 1)
plt.plot(history_no_dropout.history['accuracy'], label='No Dropout (Train)', linewidth=2)
plt.plot(history_no_dropout.history['val_accuracy'], label='No Dropout (Val)', linestyle='--', linewidth=2)
plt.plot(history_with_dropout.history['accuracy'], label='With Dropout (Train)', linewidth=2)
plt.plot(history_with_dropout.history['val_accuracy'], label='With Dropout (Val)', linestyle='--', linewidth=2)
plt.title('Accuracy Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True, alpha=0.3)

# Training Loss
plt.subplot(1, 2, 2)
plt.plot(history_no_dropout.history['loss'], label='No Dropout (Train)', linewidth=2)
plt.plot(history_no_dropout.history['val_loss'], label='No Dropout (Val)', linestyle='--', linewidth=2)
plt.plot(history_with_dropout.history['loss'], label='With Dropout (Train)', linewidth=2)
plt.plot(history_with_dropout.history['val_loss'], label='With Dropout (Val)', linestyle='--', linewidth=2)
plt.title('Loss Comparison', fontsize=14, fontweight='bold')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 6. Final Results
print("\n" + "="*50)
print("FINAL RESULTS:")
print("="*50)
print(f"No Dropout  - Train Acc: {history_no_dropout.history['accuracy'][-1]:.4f}, Val Acc: {history_no_dropout.history['val_accuracy'][-1]:.4f}")
print(f"With Dropout - Train Acc: {history_with_dropout.history['accuracy'][-1]:.4f}, Val Acc: {history_with_dropout.history['val_accuracy'][-1]:.4f}")
print(f"\nOverfitting Gap (No Dropout): {history_no_dropout.history['accuracy'][-1] - history_no_dropout.history['val_accuracy'][-1]:.4f}")
print(f"Overfitting Gap (With Dropout): {history_with_dropout.history['accuracy'][-1] - history_with_dropout.history['val_accuracy'][-1]:.4f}")

### Analysis

**What to observe:**
1. **Without Dropout**: Training accuracy quickly reaches ~100%, but validation accuracy plateaus lower → **Overfitting!**
2. **With Dropout**: Training accuracy grows slower, but validation accuracy is closer → **Better generalization!**

**The Gap**: The difference between training and validation accuracy is the "overfitting gap". Dropout reduces this gap.

**Trade-off**: Dropout slightly reduces training accuracy but significantly improves test performance. This is exactly what we want!

## 4. Hyperparameter Tuning

Modern Deep Learning has too many knobs to turn manually (Learning Rate, Units, Layers, Dropout, etc.).

### Search Strategies
-   **Grid Search**: Tries EVERY combination. Slow but exhaustive.
-   **Random Search**: Tries random combinations. Surprisingly effective and much faster.

![Grid vs Random Search](https://substackcdn.com/image/fetch/$s_!29ot!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F1bf0f3af-22ce-46b1-a0a5-d34ee4544514_772x441.png)

*Random search (right) often finds good parameters faster because it explores more unique values for each important parameter.*

## 5. Tuning with Keras Tuner

Instead of writing manual `for` loops, we use **Keras Tuner**.

*(Note: You may need to run `!pip install keras-tuner`)*

In [ ]:
# !pip install -q keras-tuner

### 4.1 Basic Keras Tuner Example (CORRECTED)

In [ ]:
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers

# 1. Define a "Hypermodel" function
# This function now accepts input_dim as a parameter to make it flexible
def build_model(hp, input_dim):
    model = keras.Sequential()
    
    # Tune the number of units in the first Dense layer
    # Choose an optimal value between 32-512
    hp_units = hp.Int('units', min_value=32, max_value=512, step=32)
    
    # Use the input_dim parameter instead of hardcoded value
    model.add(layers.Dense(units=hp_units, activation='relu', input_shape=(input_dim,)))
    model.add(layers.Dense(1, activation='sigmoid'))
    
    # Tune the learning rate for the optimizer
    # Choose an optimal value from 0.01, 0.001, or 0.0001
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    
    return model

# Example usage with actual data:
# Assuming X_train has shape (n_samples, n_features)
# input_dim = X_train.shape[1]

# 2. Initialize the Tuner (Hyperband is a popular algorithm)
# Note: You need to pass input_dim when initializing
# tuner = kt.Hyperband(
#     lambda hp: build_model(hp, input_dim=X_train.shape[1]),
#     objective='val_accuracy',
#     max_epochs=10,
#     factor=3,
#     directory='my_dir', overwrite=True,
#     project_name='intro_to_kt'
# )

# 3. Run the search with early stopping to avoid wasting time
# stop_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)
# tuner.search(X_train, y_train, epochs=50, validation_data=(X_val, y_val), callbacks=[stop_early])

# 4. Get the best hyperparameters
# best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]
# print(f"Best units: {best_hps.get('units')}")
# print(f"Best learning rate: {best_hps.get('learning_rate')}")

# 5. Build and train the best model
# best_model = tuner.hypermodel.build(best_hps)
# history = best_model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val))

### 4.2 Complete Example with Sample Data

In [ ]:
# Complete working example with synthetic data
import numpy as np
from sklearn.model_selection import train_test_split
import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers

# Generate synthetic data for demonstration
np.random.seed(42)
X = np.random.randn(1000, 10)  # 1000 samples, 10 features
y = (X[:, 0] + X[:, 1] > 0).astype(int)  # Binary classification

# Split into train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training data shape: {X_train.shape}")
print(f"Validation data shape: {X_val.shape}")

# Define hypermodel function
def build_tunable_model(hp):
    model = keras.Sequential()
    
    # Tune number of units in first layer
    hp_units = hp.Int('units', min_value=32, max_value=256, step=32)
    model.add(layers.Dense(units=hp_units, activation='relu', input_shape=(X_train.shape[1],)))
    
    # Add dropout for regularization (optional tuning)
    hp_dropout = hp.Float('dropout', min_value=0.0, max_value=0.5, step=0.1)
    model.add(layers.Dropout(hp_dropout))
    
    # Output layer
    model.add(layers.Dense(1, activation='sigmoid'))
    
    # Tune learning rate
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Initialize tuner
tuner = kt.Hyperband(
    build_tunable_model,
    objective='val_accuracy',
    max_epochs=20,
    factor=3,
    directory='my_dir', overwrite=True,
    project_name='complete_example'
)

# Display search space summary
tuner.search_space_summary()

# Run the search
stop_early = keras.callbacks.EarlyStopping(monitor='val_loss', patience=3)
tuner.search(X_train, y_train, epochs=20, validation_data=(X_val, y_val), callbacks=[stop_early], verbose=1)

# Get the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print("\n" + "="*50)
print("BEST HYPERPARAMETERS FOUND:")
print("="*50)
print(f"Units: {best_hps.get('units')}")
print(f"Dropout: {best_hps.get('dropout')}")
print(f"Learning Rate: {best_hps.get('learning_rate')}")

# Build and train the best model
best_model = tuner.hypermodel.build(best_hps)
history = best_model.fit(X_train, y_train, epochs=50, validation_data=(X_val, y_val), verbose=0)

# Evaluate the best model
val_accuracy = best_model.evaluate(X_val, y_val, verbose=0)[1]
print(f"\nBest Model Validation Accuracy: {val_accuracy:.4f}")

### 4.3 Advanced Tuning: Multiple Layers and Architecture Search

In [ ]:
# Advanced example: Tune the NUMBER of layers and their sizes
def build_advanced_model(hp):
    model = keras.Sequential()
    
    # Tune the number of layers (1 to 3)
    num_layers = hp.Int('num_layers', min_value=1, max_value=3)
    
    for i in range(num_layers):
        # Each layer can have different number of units
        hp_units = hp.Int(f'units_layer_{i}', min_value=32, max_value=256, step=32)
        
        if i == 0:
            model.add(layers.Dense(units=hp_units, activation='relu', input_shape=(X_train.shape[1],)))
        else:
            model.add(layers.Dense(units=hp_units, activation='relu'))
        
        # Optional dropout after each layer
        if hp.Boolean(f'dropout_layer_{i}'):
            model.add(layers.Dropout(0.3))
    
    # Output layer
    model.add(layers.Dense(1, activation='sigmoid'))
    
    # Tune learning rate
    hp_learning_rate = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hp_learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    
    return model

# Uncomment to run advanced tuning
# advanced_tuner = kt.RandomSearch(
#     build_advanced_model,
#     objective='val_accuracy',
#     max_trials=10,
#     directory='my_dir', overwrite=True,
#     project_name='advanced_architecture'
# )
# 
# advanced_tuner.search(X_train, y_train, epochs=20, validation_data=(X_val, y_val), verbose=1)

## 5. Exercises

**Exercise 1: Addressing the Dying ReLU**
Create a simple model but use `LeakyReLU` instead of standard `ReLU`. You can import it via `from tensorflow.keras.layers import LeakyReLU`.

**Exercise 2: Tuning the Titanic**
Load your Titanic dataset. Use **Keras Tuner** to find the best:
1. Number of layers (1, 2, or 3).
2. Dropout rate (0.2, 0.3, 0.5).
What is the best architecture found?

### Exercise 1 Solution: LeakyReLU Model

### Exercise 2 Solution: Tuning Titanic Dataset